# Lab 13: Kaggle-Style Competition — House Price Prediction with Boosting Models

**Difficulty:** ⭐⭐⭐⭐⭐ | **Estimated Time:** 5 hours

---

## Real-World Analogy First

Imagine you are a real estate analyst at Zillow. A client asks: *"What is this house worth?"* You have 20 features — square footage, location, year built, condition — and 2,000 historical sales to learn from. You try progressively smarter models: a single expert (Decision Tree), a committee of experts (Random Forest), and a series of self-correcting apprentices (Boosting). This lab replicates that exact journey, scored like a Kaggle competition where **RMSE on log-price** is the leaderboard metric.

## Competition Objectives

| # | Objective |
|---|---|
| 1 | Build a full ML pipeline from raw data to submission |
| 2 | Compare Decision Tree → Random Forest → XGBoost → LightGBM |
| 3 | Tune hyperparameters with Optuna (or random search fallback) |
| 4 | Analyze feature importance via MDI vs permutation |
| 5 | Build an experiment tracker / leaderboard table |
| 6 | Understand metric choice: RMSE vs R² vs MAE on log-price |

## Experiment Tracker
We maintain a list `experiments = []` throughout. Each entry:
```python
{'model': str, 'cv_rmse_mean': float, 'cv_rmse_std': float, 'test_rmse': float, 'tuned': bool, 'notes': str}
```

In [ ]:
# ── Section 0 ▸ Setup & Data Generation ───────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, time
from sklearn.model_selection import cross_val_score, KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.inspection import permutation_importance

warnings.filterwarnings('ignore')
np.random.seed(42)
sns.set_theme(style='whitegrid')

# Experiment tracker — all results land here
experiments = []

# Cross-validation strategy — reused everywhere for fair comparison
CV = KFold(n_splits=5, shuffle=True, random_state=42)

# Helper: RMSE scorer (negative, as sklearn convention)
def rmse_cv(model, X, y, cv=CV):
    """Return mean and std of RMSE across CV folds."""
    neg_mse = cross_val_score(model, X, y, cv=cv,
                               scoring='neg_mean_squared_error', n_jobs=-1)
    rmse_scores = np.sqrt(-neg_mse)
    return rmse_scores.mean(), rmse_scores.std()

print("Setup complete. NumPy:", np.__version__, "| Pandas:", pd.__version__)
print("Experiment tracker initialised. CV strategy: 5-fold shuffled, seed=42")

In [ ]:
# ── Section 0 ▸ Dataset Generation & EDA ──────────────────────────────────────
# We generate a realistic Kaggle-style house price dataset.
# True signal: 5 dominant features, 7 contributing, 8 pure noise.
# Target: log_price  (predicting log of sale price, as Kaggle House Prices does)

def generate_house_dataset(n_train=2000, n_test=500, seed=42):
    rng = np.random.default_rng(seed)
    N = n_train + n_test

    # ── Numerical features ────────────────────────────────────────────────────
    sqft_living   = rng.integers(500,  6000, N).astype(float)   # dominant #1
    sqft_lot      = rng.integers(1000, 40000, N).astype(float)  # contributing
    sqft_above    = (sqft_living * rng.uniform(0.6, 1.0, N)).astype(float)  # dominant #2
    sqft_basement = sqft_living - sqft_above                    # derived
    lat           = rng.uniform(47.1, 47.8, N)                  # dominant #3 (location)
    long          = rng.uniform(-122.5, -121.3, N)              # dominant #4
    yr_built      = rng.integers(1900, 2015, N).astype(float)   # contributing
    yr_renovated  = np.where(rng.random(N) < 0.2,
                             rng.integers(1980, 2020, N), 0).astype(float)  # sparse
    floors        = rng.choice([1.0, 1.5, 2.0, 2.5, 3.0], N)   # contributing
    sqft_living15 = sqft_living + rng.integers(-200, 200, N)    # correlated noise
    sqft_lot15    = sqft_lot    + rng.integers(-500, 500, N)    # correlated noise

    # ── Categorical features ──────────────────────────────────────────────────
    zipcode    = rng.choice(['98001','98002','98003','98004','98005'], N)  # dominant #5
    condition  = rng.choice([1,2,3,4,5], N)                     # contributing
    grade      = rng.choice(range(1,11), N)                     # contributing
    waterfront = rng.choice([0,1], N, p=[0.93,0.07])            # contributing
    view       = rng.choice([0,1,2,3,4], N)                     # contributing

    # ── Noise features (8 pure noise columns) ────────────────────────────────
    noise_cols = {f'noise_{i}': rng.standard_normal(N) for i in range(1, 9)}

    # ── True log-price signal ─────────────────────────────────────────────────
    zip_effect = {'98001': 0.0, '98002': 0.3, '98003': 0.6, '98004': 1.2, '98005': 0.9}
    log_price = (
        11.5
        + 0.00045 * sqft_living          # dominant
        + 0.00020 * sqft_above           # dominant
        + 1.80   * (lat - 47.1)          # dominant (location premium)
        + 0.60   * (-long - 121.3)       # dominant
        + np.array([zip_effect[z] for z in zipcode])  # dominant
        + 0.00001 * sqft_lot             # contributing
        + 0.00015 * np.maximum(sqft_basement, 0)       # contributing
        + 0.005  * grade                 # contributing
        + 0.010  * condition             # contributing
        + 0.30   * waterfront            # contributing
        + 0.05   * view                  # contributing
        + 0.0003 * (yr_built - 1900)     # contributing
        + rng.normal(0, 0.12, N)         # irreducible noise
    )

    df = pd.DataFrame({
        'sqft_living': sqft_living, 'sqft_lot': sqft_lot,
        'sqft_above': sqft_above,   'sqft_basement': sqft_basement,
        'lat': lat,                  'long': long,
        'yr_built': yr_built,        'yr_renovated': yr_renovated,
        'floors': floors,            'sqft_living15': sqft_living15,
        'sqft_lot15': sqft_lot15,
        'zipcode': zipcode,          'condition': condition,
        'grade': grade,              'waterfront': waterfront,
        'view': view,
        **noise_cols,
        'log_price': log_price
    })

    # ── Inject 8% missing values across 6 features ────────────────────────────
    missing_cols = ['sqft_basement','yr_renovated','floors','condition','view','sqft_lot15']
    for col in missing_cols:
        mask = rng.random(N) < 0.08
        df.loc[mask, col] = np.nan

    train_df = df.iloc[:n_train].reset_index(drop=True)
    test_df  = df.iloc[n_train:].reset_index(drop=True)
    return train_df, test_df

train_df, test_df = generate_house_dataset()

# ── Quick EDA ─────────────────────────────────────────────────────────────────
print(f"Train shape: {train_df.shape}  |  Test shape: {test_df.shape}")
print(f"\nMissing values (train):\n{train_df.isnull().sum()[train_df.isnull().sum()>0]}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Target distribution
axes[0].hist(train_df['log_price'], bins=40, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].set_title('Target Distribution: log_price', fontsize=13, fontweight='bold')
axes[0].set_xlabel('log_price'); axes[0].set_ylabel('Count')
axes[0].axvline(train_df['log_price'].mean(), color='red', linestyle='--', label=f"Mean={train_df['log_price'].mean():.2f}")
axes[0].legend()

# Correlation heatmap (numerical only, top correlated features)
num_cols = ['sqft_living','sqft_above','sqft_basement','lat','long',
            'yr_built','grade','condition','waterfront','view','log_price']
corr = train_df[num_cols].corr()
sns.heatmap(corr, ax=axes[1], cmap='coolwarm', center=0, annot=True,
            fmt='.2f', linewidths=0.4, annot_kws={'size':7})
axes[1].set_title('Correlation Heatmap (key features)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()
print(f"\nTop 5 correlations with log_price:")
print(corr['log_price'].drop('log_price').abs().sort_values(ascending=False).head())

---
## Section 1: Baseline — Single Decision Tree

**Analogy:** A single senior appraiser who memorises a rulebook of at most 32 questions (depth=5). Fast, interpretable, but prone to oversimplification.

We build a proper **sklearn Pipeline** here — this pipeline is reused (with model swapped) in every subsequent section, ensuring a fair apples-to-apples comparison.

In [ ]:
# ── Section 1 ▸ Preprocessing Pipeline ────────────────────────────────────────
# Identify feature types for the ColumnTransformer
TARGET = 'log_price'
CAT_COLS = ['zipcode', 'condition', 'grade', 'waterfront', 'view']
NUM_COLS = [c for c in train_df.columns if c not in CAT_COLS + [TARGET]]

print(f"Numerical features ({len(NUM_COLS)}): {NUM_COLS}")
print(f"Categorical features ({len(CAT_COLS)}): {CAT_COLS}")

# Preprocessing for numerical columns:
#   Step 1 — Impute missing values with the column median (robust to outliers)
#   Step 2 — Standardise (mean=0, std=1) so regularised models are scale-invariant
num_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

# Preprocessing for categorical columns:
#   Step 1 — Impute with the most frequent value
#   Step 2 — Ordinal encode (trees don't need one-hot; ordinal preserves integer splits)
cat_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

# Combine into a ColumnTransformer
preprocessor = ColumnTransformer([
    ('num', num_transformer, NUM_COLS),
    ('cat', cat_transformer, CAT_COLS)
], remainder='drop')

# Prepare X and y arrays
X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET].values
X_test  = test_df.drop(columns=[TARGET])
y_test  = test_df[TARGET].values

# Sanity check: fit preprocessor once to inspect output shape
X_train_proc = preprocessor.fit_transform(X_train)
print(f"\nPreprocessed train shape: {X_train_proc.shape}")
print(f"  — {len(NUM_COLS)} numerical  +  {len(CAT_COLS)} categorical = {X_train_proc.shape[1]} total features")

In [ ]:
# ── Section 1 ▸ Decision Tree — Train & Evaluate ──────────────────────────────
# Decision Tree at depth 5: max 2^5 = 32 leaf nodes.
# This is our weak baseline — high bias, low variance relative to deeper trees.

dt_model = Pipeline([
    ('prep', preprocessor),
    ('model', DecisionTreeRegressor(max_depth=5, random_state=42))
])

# 5-fold CV RMSE
dt_cv_mean, dt_cv_std = rmse_cv(dt_model, X_train, y_train)

# Final test RMSE (fit on all training data)
dt_model.fit(X_train, y_train)
dt_test_rmse = np.sqrt(mean_squared_error(y_test, dt_model.predict(X_test)))

# Log to experiment tracker
experiments.append({
    'model':        'Decision Tree',
    'cv_rmse_mean': round(dt_cv_mean, 5),
    'cv_rmse_std':  round(dt_cv_std, 5),
    'test_rmse':    round(dt_test_rmse, 5),
    'tuned':        False,
    'notes':        'depth=5'
})

print("═" * 55)
print(f"  DECISION TREE (max_depth=5)")
print(f"  CV RMSE  : {dt_cv_mean:.5f} ± {dt_cv_std:.5f}")
print(f"  Test RMSE: {dt_test_rmse:.5f}")
print("═" * 55)
print(f"\nInterpretation: A CV RMSE of {dt_cv_mean:.4f} on log_price means")
print(f"predictions are off by ≈{np.expm1(dt_cv_mean)*100:.1f}% on the original price scale.")

### Decision Tree — Interpretation

| Property | Value |
|---|---|
| Max depth | 5 |
| Max leaf nodes | 32 |
| Bias | **High** — cannot capture complex interactions |
| Variance | Moderate — depth-limited prevents overfitting |
| Interpretability | **High** — can be visualised as a flowchart |

This is the floor of our leaderboard. Every subsequent model should beat this CV RMSE.

---
## Section 2: Random Forest

**Analogy:** Instead of one senior appraiser, hire 200 appraisers who each look at a random subset of features and a random bootstrap sample of houses. Their averaged opinion is far more stable and accurate than any individual.

Key ideas: **bagging** (bootstrap aggregation) reduces variance. **Random feature subsets** at each split decorrelate the trees, amplifying the variance reduction.

In [ ]:
# ── Section 2 ▸ Random Forest — Train & CV ────────────────────────────────────
# n_estimators=200: good balance of performance vs compute for 2000 samples
# max_features='sqrt': at each split, consider sqrt(p) features — decorrelates trees
# oob_score=True: free out-of-bag validation (each tree validated on ~37% unused samples)

rf_model = Pipeline([
    ('prep', preprocessor),
    ('model', RandomForestRegressor(
        n_estimators=200,
        max_features='sqrt',
        oob_score=True,
        random_state=42,
        n_jobs=-1
    ))
])

# 5-fold CV
rf_cv_mean, rf_cv_std = rmse_cv(rf_model, X_train, y_train)

# Fit on full training data to get OOB score and test RMSE
rf_model.fit(X_train, y_train)
rf_test_rmse = np.sqrt(mean_squared_error(y_test, rf_model.predict(X_test)))

# OOB R² score (sklearn returns R², we convert for display)
rf_oob_r2 = rf_model.named_steps['model'].oob_score_

# Log experiment
experiments.append({
    'model':        'Random Forest',
    'cv_rmse_mean': round(rf_cv_mean, 5),
    'cv_rmse_std':  round(rf_cv_std, 5),
    'test_rmse':    round(rf_test_rmse, 5),
    'tuned':        False,
    'notes':        'n=200, max_features=sqrt'
})

print("═" * 55)
print(f"  RANDOM FOREST")
print(f"  CV RMSE  : {rf_cv_mean:.5f} ± {rf_cv_std:.5f}")
print(f"  Test RMSE: {rf_test_rmse:.5f}")
print(f"  OOB R²   : {rf_oob_r2:.5f}  (free validation, no extra data needed!)")
print("═" * 55)

# Improvement over baseline
dt_result = next(e for e in experiments if e['model'] == 'Decision Tree')
improvement = (dt_result['cv_rmse_mean'] - rf_cv_mean) / dt_result['cv_rmse_mean'] * 100
print(f"\n  Improvement over Decision Tree: {improvement:.1f}% reduction in CV RMSE")

In [ ]:
# ── Section 2 ▸ Feature Importances (MDI) ─────────────────────────────────────
# MDI = Mean Decrease in Impurity: sum of weighted impurity decreases at splits
# using this feature, averaged across all trees.
# Known limitation: MDI can overestimate high-cardinality and correlated features.

rf_estimator = rf_model.named_steps['model']

# Reconstruct feature names after preprocessing
feature_names_out = NUM_COLS + CAT_COLS  # ColumnTransformer preserves this order

mdi_importances = pd.Series(
    rf_estimator.feature_importances_,
    index=feature_names_out
).sort_values(ascending=True)  # ascending for horizontal bar chart

# Top 15 features
top15_mdi = mdi_importances.tail(15)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#2196F3' if 'noise' not in f else '#FF5722' for f in top15_mdi.index]
top15_mdi.plot(kind='barh', ax=ax, color=colors, edgecolor='white', alpha=0.88)
ax.set_title('Random Forest — Top 15 Feature Importances (MDI)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Mean Decrease in Impurity')

# Legend for colour coding
from matplotlib.patches import Patch
legend_elements = [Patch(color='#2196F3', label='Signal feature'),
                   Patch(color='#FF5722', label='Noise feature')]
ax.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.show()

print("\nTop 5 features by MDI:")
print(mdi_importances.tail(5).sort_values(ascending=False).to_string())
print("\nTrue dominant features: lat, long, zipcode, sqft_living, sqft_above")
print("→ Check how well MDI recovers the ground truth ordering.")

---
## Section 3: XGBoost

**Analogy:** A chain of apprentices. The first makes rough guesses; each subsequent apprentice only looks at the *mistakes* of the previous one and tries to correct them. After 500 rounds, the chain's combined corrections produce a very accurate model.

XGBoost adds: **regularisation (L1/L2)**, **second-order gradients**, and **column subsampling** to the classic gradient boosting framework — making it dramatically faster and more regularised.

In [ ]:
# ── Section 3 ▸ XGBoost — Import & Availability Check ─────────────────────────
try:
    import xgboost as xgb
    XGB_AVAILABLE = True
    print(f"XGBoost version {xgb.__version__} available.")
except ImportError:
    XGB_AVAILABLE = False
    print("XGBoost not available — using sklearn GradientBoostingRegressor as substitute.")
    print("Install with: pip install xgboost")

# Preprocessing: fit on train, transform both splits
# We need a raw numpy array for XGBoost's early stopping
X_tr_raw, X_val_raw, y_tr_raw, y_val_raw = train_test_split(
    X_train, y_train, test_size=0.15, random_state=42
)

# Fit preprocessor on training portion only (to avoid leakage)
prep_xgb = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')),
                      ('sc',  StandardScaler())]), NUM_COLS),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                      ('enc', OrdinalEncoder(handle_unknown='use_encoded_value',
                                             unknown_value=-1))]), CAT_COLS)
], remainder='drop')

X_tr_proc  = prep_xgb.fit_transform(X_tr_raw)
X_val_proc = prep_xgb.transform(X_val_raw)
X_test_proc_xgb = prep_xgb.transform(X_test)
print(f"\nTrain split: {X_tr_proc.shape}  |  Val split: {X_val_proc.shape}")

In [ ]:
# ── Section 3 ▸ XGBoost — Train with Early Stopping ──────────────────────────
# We track RMSE per boosting round to plot the learning curve.

xgb_cv_mean = xgb_cv_std = xgb_test_rmse = None  # defaults
evals_result = {}  # stores per-round metrics

if XGB_AVAILABLE:
    # Default competition params
    xgb_model_raw = xgb.XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric='rmse',
        early_stopping_rounds=40,
        verbosity=0
    )
    xgb_model_raw.fit(
        X_tr_proc, y_tr_raw,
        eval_set=[(X_tr_proc, y_tr_raw), (X_val_proc, y_val_raw)],
        verbose=False
    )
    evals_result = xgb_model_raw.evals_result()
    best_iter = xgb_model_raw.best_iteration
    print(f"Best iteration (early stopping): {best_iter}")

    # Pipeline version for fair CV comparison
    xgb_pipe = Pipeline([
        ('prep', preprocessor),
        ('model', xgb.XGBRegressor(
            n_estimators=best_iter,
            learning_rate=0.05, max_depth=5,
            subsample=0.8, colsample_bytree=0.8,
            random_state=42, verbosity=0
        ))
    ])
    xgb_cv_mean, xgb_cv_std = rmse_cv(xgb_pipe, X_train, y_train)
    xgb_pipe.fit(X_train, y_train)
    xgb_test_rmse = np.sqrt(mean_squared_error(y_test, xgb_pipe.predict(X_test)))

else:
    # Fallback: sklearn GradientBoostingRegressor
    xgb_pipe = Pipeline([
        ('prep', preprocessor),
        ('model', GradientBoostingRegressor(
            n_estimators=200, learning_rate=0.05,
            max_depth=5, subsample=0.8, random_state=42
        ))
    ])
    xgb_cv_mean, xgb_cv_std = rmse_cv(xgb_pipe, X_train, y_train)
    xgb_pipe.fit(X_train, y_train)
    xgb_test_rmse = np.sqrt(mean_squared_error(y_test, xgb_pipe.predict(X_test)))
    # Simulate evals_result for plotting
    evals_result = None

experiments.append({
    'model':        'XGBoost (default)',
    'cv_rmse_mean': round(xgb_cv_mean, 5),
    'cv_rmse_std':  round(xgb_cv_std, 5),
    'test_rmse':    round(xgb_test_rmse, 5),
    'tuned':        False,
    'notes':        'lr=0.05, depth=5, subsample=0.8' if XGB_AVAILABLE else 'sklearn GBR fallback'
})

print("═" * 55)
print(f"  XGBoost (default)")
print(f"  CV RMSE  : {xgb_cv_mean:.5f} ± {xgb_cv_std:.5f}")
print(f"  Test RMSE: {xgb_test_rmse:.5f}")
print("═" * 55)

In [ ]:
# ── Section 3 ▸ XGBoost — Learning Curve (RMSE vs Boosting Round) ─────────────
# This plot reveals: when does training RMSE plateau? When does validation diverge?
# The gap between train and val RMSE indicates overfitting.

fig, ax = plt.subplots(figsize=(10, 4))

if XGB_AVAILABLE and evals_result:
    train_rmse = evals_result['validation_0']['rmse']
    val_rmse   = evals_result['validation_1']['rmse']
    rounds = range(1, len(train_rmse) + 1)

    ax.plot(rounds, train_rmse, label='Train RMSE', color='#2196F3', lw=1.5, alpha=0.85)
    ax.plot(rounds, val_rmse,   label='Val RMSE',   color='#FF5722', lw=1.5, alpha=0.85)
    ax.axvline(best_iter, color='green', linestyle='--', lw=1.5,
               label=f'Best iteration: {best_iter}')
    ax.set_xlabel('Boosting Round')
    ax.set_ylabel('RMSE (log_price)')
    ax.set_title('XGBoost — Train vs Validation RMSE per Boosting Round',
                 fontsize=13, fontweight='bold')
    ax.legend()
    print(f"Early stopping triggered at round {best_iter}.")
    print(f"Val RMSE at best: {val_rmse[best_iter-1]:.5f}")
    print(f"Train/Val RMSE gap at best: {train_rmse[best_iter-1]:.5f} vs {val_rmse[best_iter-1]:.5f}")
else:
    # Fallback: simulate a generic learning curve for illustration
    rounds_sim = np.arange(1, 201)
    train_sim  = 0.35 * np.exp(-0.018 * rounds_sim) + 0.06 + np.random.normal(0, 0.001, 200)
    val_sim    = 0.35 * np.exp(-0.015 * rounds_sim) + 0.09 + np.random.normal(0, 0.002, 200)
    ax.plot(rounds_sim, train_sim, label='Train RMSE (simulated)', color='#2196F3')
    ax.plot(rounds_sim, val_sim,   label='Val RMSE (simulated)',   color='#FF5722')
    ax.set_title('GradientBoosting — Simulated Learning Curve (XGBoost not installed)',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Boosting Round'); ax.set_ylabel('RMSE')
    ax.legend()

ax.annotate('Overfitting begins\nwhen val diverges from train',
            xy=(0.7, 0.6), xycoords='axes fraction', fontsize=9,
            color='gray', ha='center')
plt.tight_layout()
plt.show()

---
## Section 4: LightGBM

**Analogy:** Same self-correcting apprentice chain as XGBoost, but with two key engineering shortcuts:
1. **Leaf-wise tree growth** (instead of level-wise): grow the leaf with the highest loss reduction first — fewer nodes for the same accuracy.
2. **Histogram binning**: discretise continuous features into bins, reducing gradient computation from O(n) to O(bins). Result: **3–10× faster training** on large datasets.

LightGBM also accepts **native categorical features** — no ordinal encoding needed.

In [ ]:
# ── Section 4 ▸ LightGBM — Import & Availability Check ────────────────────────
try:
    import lightgbm as lgb
    LGBM_AVAILABLE = True
    print(f"LightGBM version {lgb.__version__} available.")
except ImportError:
    LGBM_AVAILABLE = False
    print("LightGBM not available — using sklearn GradientBoostingRegressor as substitute.")
    print("Install with: pip install lightgbm")

# For LightGBM with native categoricals, we use a simpler preprocessor:
# - Only impute (no OrdinalEncoder, no StandardScaler)
# - Pass category indices to the model
if LGBM_AVAILABLE:
    # Build a preprocessor that only imputes (LightGBM handles cats natively)
    lgb_num_transformer = Pipeline([('imputer', SimpleImputer(strategy='median'))])
    lgb_cat_transformer = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                                    ('encoder', OrdinalEncoder(
                                        handle_unknown='use_encoded_value', unknown_value=-1))])
    lgb_preprocessor = ColumnTransformer([
        ('num', lgb_num_transformer, NUM_COLS),
        ('cat', lgb_cat_transformer, CAT_COLS)
    ], remainder='drop')

    # Categorical feature indices after transform (they come after NUM_COLS)
    cat_feature_indices = list(range(len(NUM_COLS), len(NUM_COLS) + len(CAT_COLS)))
    print(f"Categorical feature indices for LightGBM: {cat_feature_indices}")
else:
    lgb_preprocessor = preprocessor  # use standard preprocessor for fallback
    cat_feature_indices = None

In [ ]:
# ── Section 4 ▸ LightGBM — Train & Time Comparison ───────────────────────────
# We time LightGBM vs XGBoost training on the same data to illustrate speed diff.

lgb_cv_mean = lgb_cv_std = lgb_test_rmse = None
lgb_train_time = xgb_train_time = None

# --- Time XGBoost (re-fit for fair timing) ---
t0 = time.time()
xgb_pipe.fit(X_train, y_train)
xgb_train_time = time.time() - t0

if LGBM_AVAILABLE:
    # LightGBM doesn't support categorical_feature in Pipeline's fit_params easily,
    # so we manually preprocess and time the fit.
    X_train_lgb = lgb_preprocessor.fit_transform(X_train)
    X_test_lgb  = lgb_preprocessor.transform(X_test)

    lgb_model_raw = lgb.LGBMRegressor(
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=63,
        min_child_samples=20,
        random_state=42,
        verbose=-1
    )

    t0 = time.time()
    lgb_model_raw.fit(
        X_train_lgb, y_train,
        categorical_feature=cat_feature_indices
    )
    lgb_train_time = time.time() - t0

    lgb_test_rmse = np.sqrt(mean_squared_error(y_test, lgb_model_raw.predict(X_test_lgb)))

    # CV using Pipeline (without native cat features — fair comparison)
    lgb_pipe = Pipeline([
        ('prep', preprocessor),
        ('model', lgb.LGBMRegressor(
            n_estimators=500, learning_rate=0.05,
            num_leaves=63, min_child_samples=20,
            random_state=42, verbose=-1
        ))
    ])
    lgb_cv_mean, lgb_cv_std = rmse_cv(lgb_pipe, X_train, y_train)

else:
    # Fallback
    t0 = time.time()
    lgb_pipe = Pipeline([
        ('prep', preprocessor),
        ('model', GradientBoostingRegressor(
            n_estimators=200, learning_rate=0.05,
            max_depth=5, subsample=0.8, random_state=42
        ))
    ])
    lgb_cv_mean, lgb_cv_std = rmse_cv(lgb_pipe, X_train, y_train)
    lgb_pipe.fit(X_train, y_train)
    lgb_train_time = time.time() - t0
    lgb_test_rmse = np.sqrt(mean_squared_error(y_test, lgb_pipe.predict(X_test)))

experiments.append({
    'model':        'LightGBM',
    'cv_rmse_mean': round(lgb_cv_mean, 5),
    'cv_rmse_std':  round(lgb_cv_std, 5),
    'test_rmse':    round(lgb_test_rmse, 5),
    'tuned':        False,
    'notes':        'leaves=63, min_child=20' if LGBM_AVAILABLE else 'sklearn GBR fallback'
})

print("═" * 55)
print(f"  LightGBM")
print(f"  CV RMSE  : {lgb_cv_mean:.5f} ± {lgb_cv_std:.5f}")
print(f"  Test RMSE: {lgb_test_rmse:.5f}")
print("═" * 55)
print(f"\nTraining Time Comparison (fit on full 2000-sample train set):")
print(f"  XGBoost  : {xgb_train_time:.3f}s")
print(f"  LightGBM : {lgb_train_time:.3f}s")
if lgb_train_time and xgb_train_time and lgb_train_time > 0:
    ratio = xgb_train_time / lgb_train_time
    print(f"  Speedup  : XGBoost is {ratio:.1f}× {'slower' if ratio>1 else 'faster'} than LightGBM")

In [ ]:
# ── Section 4 ▸ Speed Comparison Bar Chart ─────────────────────────────────────
# Visualise training time and RMSE side by side for XGBoost vs LightGBM.

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: Training time
models_t = ['XGBoost', 'LightGBM']
times_t   = [xgb_train_time or 1.0, lgb_train_time or 0.33]
bar_colors = ['#FF7043', '#66BB6A']
axes[0].bar(models_t, times_t, color=bar_colors, edgecolor='white', alpha=0.9)
axes[0].set_ylabel('Training Time (seconds)')
axes[0].set_title('Training Time: XGBoost vs LightGBM', fontsize=12, fontweight='bold')
for i, v in enumerate(times_t):
    axes[0].text(i, v + 0.01, f'{v:.3f}s', ha='center', fontsize=11, fontweight='bold')

# Right: CV RMSE comparison across all models so far
exp_df = pd.DataFrame(experiments)
axes[1].barh(
    exp_df['model'], exp_df['cv_rmse_mean'],
    xerr=exp_df['cv_rmse_std'],
    color=['#90A4AE','#42A5F5','#FF7043','#66BB6A'],
    edgecolor='white', alpha=0.9, capsize=5
)
axes[1].set_xlabel('CV RMSE (log_price)')
axes[1].set_title('Model Comparison — CV RMSE so far', fontsize=12, fontweight='bold')
axes[1].invert_yaxis()  # best (lowest) at top

plt.tight_layout()
plt.show()

print("Key insight: LightGBM's histogram binning pays off most on large datasets (>100K rows).")
print("On 2000 samples, absolute time differences are small but the pattern holds.")

---
## Section 5: Optuna Hyperparameter Tuning for XGBoost

**Analogy:** Default hyperparameters are like cooking with "a pinch of salt" — fine for most dishes. Optuna is like a professional taster who systematically adjusts every ingredient, learning from each trial which direction improves the dish.

Optuna uses **Tree-structured Parzen Estimator (TPE)** — a Bayesian optimisation algorithm that builds a probabilistic model of which hyperparameter regions produced good results, then samples more from those regions.

In [ ]:
# ── Section 5 ▸ Optuna — Import & Availability Check ──────────────────────────
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    OPTUNA_AVAILABLE = True
    print(f"Optuna version {optuna.__version__} available.")
except ImportError:
    OPTUNA_AVAILABLE = False
    print("Optuna not available — using randomised search over same space (50 iterations).")
    print("Install with: pip install optuna")

# Search space definition (used by both Optuna and random search fallback)
# This defines the hyperparameter ranges we will explore.
SEARCH_SPACE = {
    'n_estimators':    (100, 1000),       # number of boosting rounds
    'max_depth':       (3, 9),            # tree depth (integer)
    'learning_rate':   (0.005, 0.3),      # log scale → focus on small values
    'subsample':       (0.5, 1.0),        # row subsampling fraction
    'colsample_bytree':(0.5, 1.0),        # column subsampling fraction
    'reg_alpha':       (1e-8, 10.0),      # L1 regularisation (log scale)
    'reg_lambda':      (1e-8, 10.0),      # L2 regularisation (log scale)
}
print("\nSearch space:")
for k, v in SEARCH_SPACE.items():
    print(f"  {k:<22}: {v}")

In [ ]:
# ── Section 5 ▸ Optuna (or Random Search) — Run 50 Trials ─────────────────────
N_TRIALS = 50
trial_numbers = []
trial_values  = []   # RMSE per trial
best_values   = []   # running best RMSE

if OPTUNA_AVAILABLE and XGB_AVAILABLE:
    # ── Optuna objective function ──────────────────────────────────────────────
    def objective(trial):
        params = {
            'n_estimators':     trial.suggest_int('n_estimators', 100, 1000),
            'max_depth':        trial.suggest_int('max_depth', 3, 9),
            'learning_rate':    trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
            'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'reg_alpha':        trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
            'reg_lambda':       trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
            'random_state': 42, 'verbosity': 0
        }
        pipe = Pipeline([
            ('prep', preprocessor),
            ('model', xgb.XGBRegressor(**params))
        ])
        mean_rmse, _ = rmse_cv(pipe, X_train, y_train)
        return mean_rmse

    study = optuna.create_study(direction='minimize',
                                sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)

    # Extract trial history for plotting
    for t in study.trials:
        trial_numbers.append(t.number)
        trial_values.append(t.value)
    best_params = study.best_params
    best_cv_rmse = study.best_value
    print(f"Optuna best CV RMSE: {best_cv_rmse:.5f}")
    print(f"Best params: {best_params}")

else:
    # ── Random search fallback ─────────────────────────────────────────────────
    rng_rs = np.random.default_rng(42)
    best_cv_rmse = np.inf
    best_params  = {}

    BaseModel = xgb.XGBRegressor if XGB_AVAILABLE else GradientBoostingRegressor

    for i in range(N_TRIALS):
        params = {
            'n_estimators':     int(rng_rs.integers(100, 1001)),
            'max_depth':        int(rng_rs.integers(3, 10)),
            'learning_rate':    float(np.exp(rng_rs.uniform(np.log(0.005), np.log(0.3)))),
            'subsample':        float(rng_rs.uniform(0.5, 1.0)),
        }
        if XGB_AVAILABLE:
            params.update({'colsample_bytree': float(rng_rs.uniform(0.5, 1.0)),
                           'reg_alpha': float(np.exp(rng_rs.uniform(np.log(1e-8), np.log(10)))),
                           'reg_lambda': float(np.exp(rng_rs.uniform(np.log(1e-8), np.log(10)))),
                           'random_state': 42, 'verbosity': 0})
        else:
            params['random_state'] = 42

        pipe = Pipeline([('prep', preprocessor), ('model', BaseModel(**params))])
        mean_rmse, _ = rmse_cv(pipe, X_train, y_train)

        trial_numbers.append(i)
        trial_values.append(mean_rmse)
        if mean_rmse < best_cv_rmse:
            best_cv_rmse = mean_rmse
            best_params  = params

        if (i + 1) % 10 == 0:
            print(f"  Trial {i+1}/{N_TRIALS} — best so far: {best_cv_rmse:.5f}")

    print(f"\nRandom search best CV RMSE: {best_cv_rmse:.5f}")
    print(f"Best params: {best_params}")

In [ ]:
# ── Section 5 ▸ Re-train Best Model & Log Result ──────────────────────────────
# Running best RMSE across trials (optimisation history)
best_values = [min(trial_values[:i+1]) for i in range(len(trial_values))]

BaseModel = xgb.XGBRegressor if XGB_AVAILABLE else GradientBoostingRegressor

tuned_pipe = Pipeline([
    ('prep', preprocessor),
    ('model', BaseModel(**best_params))
])
tuned_pipe.fit(X_train, y_train)
tuned_test_rmse = np.sqrt(mean_squared_error(y_test, tuned_pipe.predict(X_test)))

experiments.append({
    'model':        'XGBoost (tuned)',
    'cv_rmse_mean': round(best_cv_rmse, 5),
    'cv_rmse_std':  0.0,
    'test_rmse':    round(tuned_test_rmse, 5),
    'tuned':        True,
    'notes':        f'Optuna 50 trials' if OPTUNA_AVAILABLE else f'RandomSearch 50 iters'
})

# Optimisation history plot
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: All trials + running best
axes[0].scatter(trial_numbers, trial_values, alpha=0.4, s=20,
                color='#90A4AE', label='Trial RMSE')
axes[0].plot(trial_numbers, best_values, color='#E53935', lw=2.0,
             label='Best so far')
axes[0].set_xlabel('Trial Number')
axes[0].set_ylabel('CV RMSE')
method = 'Optuna TPE' if OPTUNA_AVAILABLE else 'Random Search'
axes[0].set_title(f'{method} — Optimisation History ({N_TRIALS} trials)',
                  fontsize=12, fontweight='bold')
axes[0].legend()

# Right: Default vs Tuned
xgb_default_rmse = next(e['cv_rmse_mean'] for e in experiments if e['model']=='XGBoost (default)')
comparison_vals = [xgb_default_rmse, best_cv_rmse]
comparison_labs = ['XGBoost\n(default)', 'XGBoost\n(tuned)']
bars = axes[1].bar(comparison_labs, comparison_vals,
                   color=['#FF7043','#43A047'], edgecolor='white', alpha=0.9, width=0.4)
for bar, val in zip(bars, comparison_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.0002,
                 f'{val:.5f}', ha='center', fontsize=11, fontweight='bold')
axes[1].set_ylabel('CV RMSE')
axes[1].set_title('Default vs Tuned XGBoost', fontsize=12, fontweight='bold')
pct_gain = (xgb_default_rmse - best_cv_rmse) / xgb_default_rmse * 100
axes[1].text(0.5, 0.85, f'Improvement: {pct_gain:.2f}%',
             transform=axes[1].transAxes, ha='center', fontsize=11, color='green')

plt.tight_layout()
plt.show()

print(f"Default XGBoost CV RMSE : {xgb_default_rmse:.5f}")
print(f"Tuned XGBoost  CV RMSE  : {best_cv_rmse:.5f}")
print(f"Improvement             : {pct_gain:.2f}%")
print(f"Tuned Test RMSE         : {tuned_test_rmse:.5f}")

---
## Section 6: Feature Importance Analysis

**Two methods, two stories:**

| Method | Mechanism | Bias | Speed |
|---|---|---|---|
| MDI (Mean Decrease Impurity) | Sum of split gain weighted by node samples | Favours high-cardinality & correlated features | Very fast (computed during training) |
| Permutation Importance | Measure RMSE increase when feature is randomly shuffled | More reliable, model-agnostic | Slow (5 shuffles × n_features × n_CV_passes) |

Since we **know the ground truth** (5 dominant features), we can check which method recovers them more accurately.

In [ ]:
# ── Section 6 ▸ MDI Importance from Random Forest ─────────────────────────────
# We already computed this in Section 2; here we format it for side-by-side display.

rf_estimator = rf_model.named_steps['model']
feature_names_out = NUM_COLS + CAT_COLS

mdi_series = pd.Series(
    rf_estimator.feature_importances_,
    index=feature_names_out
).sort_values(ascending=False)

print("MDI Importance — Top 15:")
print(mdi_series.head(15).to_string())

# Ground truth: these 5 should dominate
TRUE_DOMINANT = {'lat', 'long', 'zipcode', 'sqft_living', 'sqft_above'}
top5_mdi = set(mdi_series.head(5).index)

print(f"\nGround-truth dominant features : {sorted(TRUE_DOMINANT)}")
print(f"MDI top-5 recovered            : {sorted(top5_mdi)}")
print(f"Overlap                        : {len(top5_mdi & TRUE_DOMINANT)}/5 features")

In [ ]:
# ── Section 6 ▸ Permutation Importance — Best Model ──────────────────────────
# We use the tuned XGBoost (or RF if XGB unavailable) as our best model.
# Permutation importance: for each feature, shuffle its values and measure
# how much the model's test RMSE increases. A large increase = important feature.

# Use preprocessed test data for permutation importance
X_test_proc_perm = preprocessor.fit_transform(X_train)  # fit on train
X_test_proc_perm2 = preprocessor.transform(X_test)      # transform test

# Choose best model estimator (fitted on full train)
best_model_for_perm = tuned_pipe.named_steps['model']

print("Computing permutation importances (n_repeats=5) — may take 20-60s ...")
perm_result = permutation_importance(
    best_model_for_perm, X_test_proc_perm2, y_test,
    n_repeats=5, random_state=42, n_jobs=-1,
    scoring='neg_mean_squared_error'
)
# Convert neg_MSE decrease → positive RMSE-scale importance
perm_series = pd.Series(
    perm_result.importances_mean,
    index=feature_names_out
).sort_values(ascending=False)

# Side-by-side bar chart: MDI vs Permutation
top_n = 15
top15_features = list(dict.fromkeys(
    list(mdi_series.head(top_n).index) + list(perm_series.head(top_n).index)
))[:top_n]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, series, title in zip(
    axes,
    [mdi_series[top15_features], perm_series[top15_features]],
    ['MDI Importance (Random Forest)', 'Permutation Importance (Best Model)']
):
    sorted_s = series.sort_values(ascending=True)
    colors = ['#E53935' if f in TRUE_DOMINANT else
              '#FF7043' if 'noise' not in f else '#90A4AE'
              for f in sorted_s.index]
    sorted_s.plot(kind='barh', ax=ax, color=colors, edgecolor='white', alpha=0.85)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Importance Score')

from matplotlib.patches import Patch
legend_els = [
    Patch(color='#E53935', label='True dominant feature'),
    Patch(color='#FF7043', label='Contributing feature'),
    Patch(color='#90A4AE', label='Noise feature'),
]
axes[1].legend(handles=legend_els, loc='lower right', fontsize=9)

plt.suptitle('MDI vs Permutation Importance — Which method finds the true signal?',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

top5_perm = set(perm_series.head(5).index)
print(f"\nPermutation top-5 recovered: {sorted(top5_perm)}")
print(f"Overlap with ground truth   : {len(top5_perm & TRUE_DOMINANT)}/5 features")
print(f"\nConclusion: {'Permutation' if len(top5_perm & TRUE_DOMINANT) >= len(top5_mdi & TRUE_DOMINANT) else 'MDI'} "
      f"recovers more ground-truth features on this dataset.")

---
## Section 7: Experiment Tracker & Leaderboard

A proper Kaggle competitor maintains a structured experiment log. Every model run, every hyperparameter change, every preprocessing tweak gets recorded. This prevents the common mistake of "losing" a good model configuration and allows systematic retrospection.

In [ ]:
# ── Section 7 ▸ Leaderboard Table ─────────────────────────────────────────────
# Compile all experiments into a styled DataFrame for display.

lb = pd.DataFrame(experiments)
lb = lb.sort_values('cv_rmse_mean').reset_index(drop=True)
lb.index = lb.index + 1  # rank starts at 1
lb.index.name = 'Rank'

# Add delta column: how much better vs Decision Tree baseline
baseline_rmse = lb.loc[lb['model'] == 'Decision Tree', 'cv_rmse_mean'].values[0]
lb['delta_vs_baseline'] = ((baseline_rmse - lb['cv_rmse_mean']) / baseline_rmse * 100).round(2)

# Display
pd.set_option('display.float_format', '{:.5f}'.format)
print("=" * 85)
print(f"{'COMPETITION LEADERBOARD':^85}")
print("=" * 85)
print(lb[['model','cv_rmse_mean','cv_rmse_std','test_rmse','tuned','delta_vs_baseline','notes']].to_string())
print("=" * 85)

winner = lb.iloc[0]
print(f"\nWINNER: {winner['model']}")
print(f"  CV RMSE  : {winner['cv_rmse_mean']:.5f} ± {winner['cv_rmse_std']:.5f}")
print(f"  Test RMSE: {winner['test_rmse']:.5f}")
print(f"  Notes    : {winner['notes']}")
print(f"\nImprovement over Decision Tree baseline: {winner['delta_vs_baseline']:.2f}%")

In [ ]:
# ── Section 7 ▸ Leaderboard Visualisation ─────────────────────────────────────
# Bar chart with error bars showing CV RMSE for all models.
# The lower the bar, the better the model.

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left: CV RMSE with error bars
model_order = lb.sort_values('cv_rmse_mean')['model'].tolist()
rmse_vals   = lb.sort_values('cv_rmse_mean')['cv_rmse_mean'].tolist()
rmse_stds   = lb.sort_values('cv_rmse_mean')['cv_rmse_std'].tolist()

palette = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(model_order)))
bars = axes[0].barh(
    model_order[::-1], rmse_vals[::-1],
    xerr=rmse_stds[::-1],
    color=palette[::-1], edgecolor='white', alpha=0.9, capsize=5
)
axes[0].set_xlabel('CV RMSE (log_price) — lower is better')
axes[0].set_title('Model Leaderboard — CV RMSE with Std Bars',
                  fontsize=12, fontweight='bold')
# Annotate bars
for bar, val in zip(bars, rmse_vals[::-1]):
    axes[0].text(val + 0.0002, bar.get_y() + bar.get_height()/2,
                 f'{val:.5f}', va='center', fontsize=9)

# Right: CV RMSE vs Test RMSE scatter — detect generalisation gaps
for _, row in lb.iterrows():
    axes[1].scatter(row['cv_rmse_mean'], row['test_rmse'], s=120, zorder=5)
    axes[1].annotate(row['model'], (row['cv_rmse_mean'], row['test_rmse']),
                     textcoords='offset points', xytext=(5, 5), fontsize=8)

# Diagonal line: perfect generalisation (CV RMSE = Test RMSE)
all_rmse = lb['cv_rmse_mean'].tolist() + lb['test_rmse'].tolist()
lim = (min(all_rmse)*0.98, max(all_rmse)*1.02)
axes[1].plot(lim, lim, 'k--', alpha=0.3, label='CV = Test (perfect)')
axes[1].set_xlabel('CV RMSE')
axes[1].set_ylabel('Test RMSE')
axes[1].set_title('CV RMSE vs Test RMSE — Generalisation Check',
                  fontsize=12, fontweight='bold')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

print("Points above the diagonal → test RMSE > CV RMSE → slight overfitting to CV folds.")
print("Points on/below diagonal → model generalises well.")

In [ ]:
# ── Section 7 ▸ Leaderboard Insights ──────────────────────────────────────────
# Quantify: how much did each upgrade improve over the previous?

lb_sorted = lb.sort_values('cv_rmse_mean').reset_index(drop=True)

print("Marginal gains across the upgrade path:")
print("-" * 60)

# Find specific models
def get_rmse(name):
    row = lb[lb['model'] == name]
    return row['cv_rmse_mean'].values[0] if len(row) > 0 else None

dt_r  = get_rmse('Decision Tree')
rf_r  = get_rmse('Random Forest')
xgb_r = get_rmse('XGBoost (default)')
lgb_r = get_rmse('LightGBM')
tun_r = get_rmse('XGBoost (tuned)')

steps = [
    ('Decision Tree → Random Forest',    dt_r,  rf_r),
    ('Random Forest → XGBoost (default)',rf_r,  xgb_r),
    ('XGBoost → LightGBM',               xgb_r, lgb_r),
    ('XGBoost default → XGBoost tuned',  xgb_r, tun_r),
]
for label, before, after in steps:
    if before is not None and after is not None:
        gain = (before - after) / before * 100
        arrow = '↑ better' if gain > 0 else '↓ worse'
        print(f"  {label:<42}: {gain:+.2f}% ({arrow})")

print("-" * 60)
print(f"\nTotal improvement (DT → best): {(dt_r - lb_sorted.iloc[0]['cv_rmse_mean'])/dt_r*100:.1f}%")
print("\nKey takeaway: ensemble methods offer large gains over single trees;")
print("hyperparameter tuning offers smaller marginal gains (diminishing returns).")

---
## Section 8: Metric Choice Discussion

**Key question:** You want to be in the **top 30% on Kaggle**. The leaderboard sorts by RMSE on log-price. But your stakeholder wants R². Does optimising RMSE guarantee a good R²?

**Real Kaggle House Prices competition:** Uses RMSLE (Root Mean Squared Log Error) on original price. When the target *is already* `log_price`, RMSLE ≈ RMSE on `log_price`. We verify this below.

In [ ]:
# ── Section 8 ▸ Multi-Metric Comparison ──────────────────────────────────────
# Compute RMSE, R², MAE for all models on the test set.
# Show that different metrics can rank models differently.

# Re-fit all pipelines on full training data (some may already be fit)
all_pipes = [
    ('Decision Tree',      dt_model),
    ('Random Forest',      rf_model),
    ('XGBoost (default)',  xgb_pipe),
    ('XGBoost (tuned)',    tuned_pipe),
]
if LGBM_AVAILABLE:
    # LightGBM was fit outside a standard pipeline; re-create for fair comparison
    lgb_compare = Pipeline([
        ('prep', preprocessor),
        ('model', lgb.LGBMRegressor(
            n_estimators=300, learning_rate=0.05,
            num_leaves=63, min_child_samples=20,
            random_state=42, verbose=-1
        ))
    ])
    lgb_compare.fit(X_train, y_train)
    all_pipes.append(('LightGBM', lgb_compare))
else:
    all_pipes.append(('LightGBM (GBR)', lgb_pipe))

metric_rows = []
for name, pipe in all_pipes:
    pipe.fit(X_train, y_train)   # ensure fitted
    y_pred = pipe.predict(X_test)

    rmse_val = np.sqrt(mean_squared_error(y_test, y_pred))
    r2_val   = r2_score(y_test, y_pred)
    mae_val  = mean_absolute_error(y_test, y_pred)

    # RMSLE on original price = RMSE on log price (since target = log_price)
    # Verify: exp(log_price) = price; RMSLE(price) = sqrt(mean((log(pred_price)-log(true_price))^2))
    #       = sqrt(mean((log_pred - log_true)^2)) = RMSE on log_price
    rmsle_equiv = rmse_val  # by definition when target is already log-transformed

    metric_rows.append({
        'Model': name,
        'RMSE (log)': round(rmse_val, 5),
        'R²': round(r2_val, 5),
        'MAE (log)': round(mae_val, 5),
        'RMSLE (≡RMSE)': round(rmsle_equiv, 5)
    })

metrics_df = pd.DataFrame(metric_rows).set_index('Model')

# Rank by each metric
metrics_df['Rank by RMSE'] = metrics_df['RMSE (log)'].rank().astype(int)
metrics_df['Rank by R²']   = metrics_df['R²'].rank(ascending=False).astype(int)
metrics_df['Rank by MAE']  = metrics_df['MAE (log)'].rank().astype(int)

print(metrics_df.to_string())
print("\nNote: RMSLE on original price ≡ RMSE on log_price — identical column!")
print("This is the mathematical reason the Kaggle competition uses log-transformed prices.")

In [ ]:
# ── Section 8 ▸ Metric Divergence Visualisation ───────────────────────────────
# Show radar/parallel plot of model ranks across metrics to reveal disagreements.

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

metric_pairs = [
    ('RMSE (log)', 'R²',       'RMSE vs R²'),
    ('RMSE (log)', 'MAE (log)', 'RMSE vs MAE'),
    ('R²',         'MAE (log)', 'R² vs MAE'),
]

colors_m = plt.cm.tab10(np.linspace(0, 0.8, len(metrics_df)))

for ax, (xm, ym, title) in zip(axes, metric_pairs):
    for i, (model_name, row) in enumerate(metrics_df.iterrows()):
        ax.scatter(row[xm], row[ym], s=100, color=colors_m[i], zorder=5, label=model_name)
        ax.annotate(model_name.replace(' ', '\n'),
                    (row[xm], row[ym]),
                    textcoords='offset points', xytext=(5, 5),
                    fontsize=7, color=colors_m[i])
    ax.set_xlabel(xm, fontsize=10)
    ax.set_ylabel(ym, fontsize=10)
    ax.set_title(title, fontsize=11, fontweight='bold')
    # For RMSE/MAE: lower is better (invert x-axis when RMSE is x)
    if 'RMSE' in xm or 'MAE' in xm:
        ax.invert_xaxis()
    if 'RMSE' in ym or 'MAE' in ym:
        ax.invert_yaxis()

plt.suptitle('Metric Comparison: Do Different Metrics Agree on Rankings?',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# Rank correlation between metrics
from scipy.stats import spearmanr
r_rmse_r2, _ = spearmanr(metrics_df['Rank by RMSE'], metrics_df['Rank by R²'])
r_rmse_mae, _ = spearmanr(metrics_df['Rank by RMSE'], metrics_df['Rank by MAE'])
print(f"Spearman rank correlation:")
print(f"  RMSE vs R²  rankings: ρ = {r_rmse_r2:.3f}")
print(f"  RMSE vs MAE rankings: ρ = {r_rmse_mae:.3f}")
print()
print("Discussion: For Kaggle top-30%, always optimise the exact leaderboard metric (RMSE on log).")
print("R² is useful for reporting to stakeholders but may disagree with RMSE rankings when")
print("models differ in how they handle outlier predictions (R² is more sensitive to outliers).")
print()
print("RMSLE identity: RMSLE(price) = RMSE(log_price) — shown because:")
print("  RMSLE = sqrt(mean((log(1+pred) - log(1+true))^2))")
print("  When target is already log_price: log(1+price) ≈ log(price) for large prices")
print("  Exact equivalence holds when target = log(price) and predictions are in log space.")

---
## Section 9: Self-Check

Test your understanding. Answer each question before expanding the hint.

---

**Q1.** Random Forest and XGBoost both get similar CV RMSE. What does this imply about the bias-variance tradeoff for each?

<details>
<summary>Show Answer</summary>

Similar CV RMSE does **not** mean the models are similar internally. Random Forest reduces variance via bagging and averaging — it starts with low-bias deep trees and averages out their variance. XGBoost reduces bias sequentially — it starts with high-bias shallow trees and iteratively corrects errors. Both can arrive at the same total error (bias² + variance + irreducible) through different paths. The key implication: if you have more data, XGBoost often improves faster because its sequential structure allows it to capture complex interactions with more signal; Random Forest's averaging benefit plateaus once it has enough trees. Check their error decompositions separately rather than concluding they're interchangeable.

</details>

---

**Q2.** Your tuned XGBoost (Optuna, 50 trials) is 0.3% better than default. Is this likely a real improvement or noise?

<details>
<summary>Show Answer</summary>

Almost certainly **noise** for a 2,000-sample dataset. With 5-fold CV, each fold has ~400 validation samples. The standard error of RMSE across 5 folds is typically several times the 0.3% gap. Run a proper significance test: if the tuned model's CV RMSE distribution (across 5 folds) overlaps substantially with the default model's, the difference is not reliable. On a 2K dataset, Optuna's Bayesian search has limited signal because the 5-fold CV estimates themselves have high variance. The gain becomes meaningful on 100K+ samples where CV estimates stabilise. Rule of thumb: treat gains below 1 CV std as noise; gains above 2 CV std as potentially real.

</details>

---

**Q3.** LightGBM trains 3× faster than XGBoost on this 2,000-sample dataset. When would this speedup matter most?

<details>
<summary>Show Answer</summary>

The 3× speedup on 2,000 samples is **almost irrelevant** in absolute terms (seconds vs milliseconds). LightGBM's architectural advantages — histogram binning (O(bins) instead of O(n) gradient computation), leaf-wise growth, and sparse-aware algorithms — compound with scale. The speedup matters most when: (1) **dataset size > 100K rows** where XGBoost's exact split finding becomes a bottleneck; (2) **high-cardinality categorical features** that LightGBM handles natively; (3) **memory-constrained environments** since LightGBM uses less RAM via histogram approximation; (4) **rapid prototyping** with Optuna where you need thousands of trials — a 3× per-trial speedup translates to 3× more trials in the same wall clock time.

</details>

---

**Q4.** Feature importance shows `sqft_living` is #1 by MDI but #3 by permutation. What might explain the discrepancy?

<details>
<summary>Show Answer</summary>

Two main causes: (1) **Correlation with `sqft_above`**: MDI assigns importance to whichever correlated feature gets used at a split first — Random Forest trees may frequently split on `sqft_living` early, giving it high MDI, while `sqft_above` (which carries similar signal) absorbs some importance later. Permutation importance, when `sqft_living` is shuffled, sees only a partial drop because `sqft_above` (still intact) compensates — artificially lowering its permutation rank. (2) **MDI bias toward continuous high-cardinality features**: `sqft_living` has thousands of unique values (many split candidates), so it appears frequently in splits purely due to combinatorial advantage, inflating its MDI. Permutation importance is immune to this since it measures actual predictive contribution. The ground truth is better approximated by permutation importance in this case.

</details>

---

**Q5.** You're targeting top 30% on Kaggle. Your CV RMSE is stable but test RMSE is always 15% worse. What is the most likely cause?

<details>
<summary>Show Answer</summary>

A consistent 15% gap between CV and public leaderboard RMSE, with stable (low-variance) CV, is the classic signature of **distribution shift** — not overfitting. Overfitting would cause high CV variance and model-dependent gaps. Distribution shift means the public test set was drawn from a different distribution than training (e.g., different time period, different geographic region, different data collection process). Remedies: (1) Check if test set features have different summary statistics (mean, variance, missing rate) than training — a quick `describe()` comparison; (2) Use **adversarial validation**: train a classifier to distinguish train vs test rows; if AUC > 0.6, there's meaningful shift; (3) Re-weight training samples to match test distribution; (4) Use temporal CV if data has time structure (never use random CV on time-series data). A 15% consistent gap is a data problem, not a model problem.

</details>